In [6]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
 
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df_clean = df.copy()
print(f"Yüklendi: {df_clean.shape}")

Yüklendi: (7043, 21)


In [7]:
# customerID analiz için gereksiz
df_clean.drop(columns=["customerID"], inplace=True)
 
# TotalCharges object → numeric (boşluk = NaN)
df_clean["TotalCharges"] = pd.to_numeric(
    df_clean["TotalCharges"], errors="coerce")
 
null_mask = df_clean["TotalCharges"].isna()
print(f"TotalCharges eksik: {null_mask.sum()} satır")
print("Bu satırların tenure değerleri:",
      df_clean.loc[null_mask, "tenure"].values)
 
# tenure=0 olan yeni müşteriler → TotalCharges=0
df_clean["TotalCharges"].fillna(0, inplace=True)
 
# Churn → binary integer (Yes=1, No=0)
df_clean["Churn"] = (df_clean["Churn"] == "Yes").astype(int)
 
print(f"Temizlik tamam: {df_clean.shape}")
print(df_clean["Churn"].value_counts())

TotalCharges eksik: 11 satır
Bu satırların tenure değerleri: [0 0 0 0 0 0 0 0 0 0 0]
Temizlik tamam: (7043, 20)
Churn
0    5174
1    1869
Name: count, dtype: int64


C:\Users\ferha\AppData\Local\Temp\ipykernel_8928\3178924464.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean["TotalCharges"].fillna(0, inplace=True)


In [8]:
# 1) Ortalama aylık ücret
df_clean["AvgMonthlyCharge"] = np.where(
    df_clean["tenure"] > 0,
    df_clean["TotalCharges"] / df_clean["tenure"],
    df_clean["MonthlyCharges"]
)
 
# 2) Sahip olunan internet hizmeti sayısı
internet_services = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]
for col in internet_services:
    df_clean[col + "_bin"] = (df_clean[col] == "Yes").astype(int)
 
df_clean["NumInternetServices"] = df_clean[
    [c + "_bin" for c in internet_services]
].sum(axis=1)
df_clean.drop(columns=[c + "_bin" for c in internet_services], inplace=True)
 
# 3) Yüksek aylık ücret bayrağı (üst çeyrek)
q75 = df_clean["MonthlyCharges"].quantile(0.75)
df_clean["HighMonthlyCharge"] = (
    df_clean["MonthlyCharges"] > q75).astype(int)
 
print("Yeni özellikler:")
print(df_clean[[
    "AvgMonthlyCharge", "NumInternetServices", "HighMonthlyCharge"
]].describe().round(2))

Yeni özellikler:
       AvgMonthlyCharge  NumInternetServices  HighMonthlyCharge
count           7043.00              7043.00            7043.00
mean              64.76                 2.04               0.25
std               30.19                 1.85               0.43
min               13.78                 0.00               0.00
25%               35.94                 0.00               0.00
50%               70.34                 2.00               0.00
75%               90.17                 3.00               0.00
max              121.40                 6.00               1.00


In [9]:
# Binary kategorik → 0/1
binary_cols = [
    "gender", "Partner", "Dependents",
    "PhoneService", "PaperlessBilling"
]
for col in binary_cols:
    df_clean[col] = LabelEncoder().fit_transform(df_clean[col])
 
# Çok sınıflı kategorik → One-Hot Encoding
ohe_cols = [
    "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies", "Contract", "PaymentMethod"
]
df_model = pd.get_dummies(df_clean, columns=ohe_cols, drop_first=True)
 
print(f"Encoding sonrası özellik sayısı: {df_model.shape[1] - 1}")


Encoding sonrası özellik sayısı: 33


In [10]:
# İşlenmiş veriyi kaydet
df_model.to_csv("../data/telco_model_ready.csv", index=False)
print("Kaydedildi: data/telco_model_ready.csv")
print(f"Boyut: {df_model.shape}")

Kaydedildi: data/telco_model_ready.csv
Boyut: (7043, 34)
